# Agente Funcional - Reglamento Academico Duoc UC
## Evaluacion Parcial N°2 - Ingenieria de Soluciones con IA (ISY0101)
**Autor:** Jaime Delgadillo Lopez  
**Fecha:** Junio 2026

Este notebook construye un **agente** sobre el sistema RAG desarrollado en la EP1.
La diferencia con EP1 es que ahora el sistema tiene:
- **Memoria**: recuerda lo que se ha preguntado antes en la conversacion
- **Herramientas**: puede buscar en el reglamento Y responder preguntas generales
- **Planificacion**: decide automaticamente que herramienta usar segun la pregunta

In [ ]:
# ============================================================
# CELDA 1 - INSTALACION DE LIBRERIAS
# ============================================================
# Ejecutar SOLO la primera vez y luego reiniciar el kernel.

!pip install -U langchain langchain-core langchain-openai langchain-classic sentence-transformers faiss-cpu pdfplumber

# o desde tarminal pip install -U langchain langchain-core langchain-openai langchain-classic sentence-transformers faiss-cpu pdfplumber

# Librerias utilizadas en el proyecto:
#
# - langchain              -> framework principal de agentes
# - langchain-core         -> prompts, mensajes y tools
# - langchain-openai       -> conexion con modelos GPT/OpenAI
# - langchain-classic      -> AgentExecutor y agentes clasicos
# - sentence-transformers  -> generacion de embeddings
# - faiss-cpu              -> base vectorial para busqueda semantica
# - pdfplumber             -> extraccion de texto desde PDFs

In [ ]:
# ============================================================
# CELDA 2 - IMPORTACIONES Y CONFIGURACION DEL MODELO
# ============================================================
# Importamos todo lo que necesitamos para que el agente funcione
import os
import re
import pdfplumber
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    AIMessage
)
from langchain_core.tools import tool
# IMPORTS NUEVOS
from langchain_classic.agents import AgentExecutor
from langchain_classic.agents import create_tool_calling_agent
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

# CONFIGURACION MODELO

os.environ["GITHUB_TOKEN"] = "Su GitHub_Token ACA"  #Copiar el token aca, por motivos de seguridad no se debe subir al repositorio.

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.environ["GITHUB_TOKEN"],
    base_url="https://models.inference.ai.azure.com",
    temperature=0.2
)

print("Modelo configurado correctamente")

In [ ]:
# ============================================================
# CARGAR Y PREPARAR REGLAMENTO   CELDA 3
# ============================================================

import pdfplumber
import re
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

# Ruta PDF
#ruta_pdf = r"C:\Reglamento_Academico_Duoc\RES-VRA-03-2024-NUEVO-REGLAMENTO-ACADÉMICO63-1.pdf" # En caso de error de lectura del reglamento, descargar y definir la ruta.
ruta_pdf = "RES-VRA-03-2024-NUEVO-REGLAMENTO-ACADEMICO63-1.pdf"
# ============================================================
# EXTRAER TEXTO
# ============================================================

print("Paso 1: Extrayendo texto del PDF...")
texto_completo = ""
with pdfplumber.open(ruta_pdf) as pdf:

    for pagina in pdf.pages:

        texto = pagina.extract_text()

        if texto:
            texto_completo += texto + "\n"

print(f"Texto extraido: {len(texto_completo)} caracteres")

# ============================================================
# DIVIDIR EN ARTICULOS
# ============================================================

print("Paso 2: Dividiendo en articulos...")
patron = r'(Art[ií]culo\s*(?:N°|Nº|No\.?|)?\s*\d+)'
partes = re.split(patron, texto_completo)
fragmentos = []
MAX_CHARS = 800

for i in range(1, len(partes), 2):

    if i + 1 < len(partes):

        titulo = partes[i].strip()
        contenido = partes[i + 1].strip()

        if len(contenido) > MAX_CHARS:
            contenido = contenido[:MAX_CHARS] + "..."

        fragmento = f"{titulo}\n{contenido}"

        fragmentos.append(fragmento)

print(f"Total articulos encontrados: {len(fragmentos)}")

# Validacion

if len(fragmentos) == 0:
    print("ERROR: No se detectaron articulos")

# ============================================================
# EMBEDDINGS
# ============================================================

print("Paso 3: Generando embeddings...")

modelo_embeddings = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

embeddings = modelo_embeddings.encode(
    fragmentos,
    show_progress_bar=True
)

embeddings = np.array(embeddings).astype('float32')

print(f"Shape embeddings: {embeddings.shape}")

# ============================================================
# FAISS
# ============================================================

print("Paso 4: Construyendo indice FAISS...")
dimension = embeddings.shape[1]
indice_faiss = faiss.IndexFlatL2(dimension)
indice_faiss.add(embeddings)
print(f"Base vectorial lista: {indice_faiss.ntotal} articulos")

In [ ]:
# ============================================================
# CELDA 4 - HERRAMIENTAS
# ============================================================

from langchain_core.tools import tool

# ============================================================
# TOOL RAG
# ============================================================

@tool
def buscar_en_reglamento(pregunta: str) -> str:
    """
    Busca informacion en el Reglamento Academico de Duoc UC.
    """

    # Embedding pregunta
    embedding_pregunta = modelo_embeddings.encode([pregunta])

    embedding_pregunta = np.array(
        embedding_pregunta
    ).astype('float32')

    # Buscar similares
    distancias, indices = indice_faiss.search(
        embedding_pregunta,
        k=3
    )

    resultados = []

    for distancia, idx in zip(distancias[0], indices[0]):

        if idx < len(fragmentos):

            resultados.append(fragmentos[idx])

    contexto = "\n\n---\n\n".join(resultados)

    return f"""
Usa EXCLUSIVAMENTE la siguiente informacion del reglamento academico
para responder la pregunta.

Si el reglamento no contiene la informacion,
indica que no fue encontrada.

CONTEXTO:

{contexto}
"""


# ============================================================
# TOOL GENERAL
# ============================================================

@tool
def consulta_general(pregunta: str) -> str:
    """
    Responde preguntas generales educativas.
    """

    return f"""
Pregunta general del estudiante:

{pregunta}
"""
# ============================================================
# LISTA TOOLS
# ============================================================

tools = [
    buscar_en_reglamento,
    consulta_general
]

print("Herramientas configuradas:")

for t in tools:
    print("-", t.name)

In [ ]:
# ============================================================
# CELDA 5 - CONSTRUCCION DEL AGENTE CON MEMORIA
# ============================================================

from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

from langchain_classic.agents import (
    AgentExecutor,
    create_tool_calling_agent
)

# ============================================================
# PROMPT DEL AGENTE
# ============================================================

prompt_agente = ChatPromptTemplate.from_messages([

    (
        "system",
        """
Eres un asistente academico especializado en el Reglamento Academico de Duoc UC
(Resolucion N°03/2024).

Tienes acceso a dos herramientas:

1. buscar_en_reglamento
   - Usar para preguntas sobre:
     notas, asistencia, eliminacion, evaluaciones,
     reglamentos, articulos, titulacion y normas academicas.

2. consulta_general
   - Usar para preguntas generales sobre educacion superior
     que NO pertenezcan especificamente al reglamento.

REGLAS IMPORTANTES:

- Siempre usa una herramienta antes de responder.
- Nunca inventes informacion ni articulos.
- Si no encuentras informacion suficiente, dilo claramente.
- Cuando uses informacion del reglamento,
  cita el articulo correspondiente.
- Usa un tono claro, amable y sencillo.
- Puedes considerar el historial previo de la conversacion.
"""
    ),

    # ========================================================
    # MEMORIA CONVERSACIONAL
    # ========================================================

    MessagesPlaceholder(
        variable_name="historial_chat"
    ),

    # ========================================================
    # PREGUNTA USUARIO
    # ========================================================

    (
        "human",
        "{pregunta_usuario}"
    ),

    # ========================================================
    # RAZONAMIENTO INTERNO AGENTE
    # ========================================================

    MessagesPlaceholder(
        variable_name="agent_scratchpad"
    ),
])

# ============================================================
# CREAR AGENTE
# ============================================================

agente = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt_agente
)

# ============================================================
# EJECUTOR DEL AGENTE
# ============================================================

ejecutor_agente = AgentExecutor(
    agent=agente,
    tools=tools,
    verbose=True,
    max_iterations=5
)

print("Agente construido y listo")

In [ ]:
# ============================================================
# CELDA 6 - SISTEMA DE MEMORIA
# ============================================================

from langchain_core.messages import (
    HumanMessage,
    AIMessage
)

# ============================================================
# MEMORIA
# ============================================================

# Memoria conversacional corta
historial_conversacion = []

# Memoria larga simple
temas_consultados = []

# Limite historial
MAX_MENSAJES = 10


# ============================================================
# FUNCION PRINCIPAL
# ============================================================

def preguntar_al_agente(pregunta):

    global historial_conversacion
    global temas_consultados

    try:

        # ====================================================
        # INVOCAR AGENTE
        # ====================================================

        resultado = ejecutor_agente.invoke({

            "pregunta_usuario": pregunta,

            "historial_chat": historial_conversacion

        })

        respuesta = resultado["output"]

        # ====================================================
        # GUARDAR MEMORIA CORTA
        # ====================================================

        historial_conversacion.append(
            HumanMessage(content=pregunta)
        )

        historial_conversacion.append(
            AIMessage(content=respuesta)
        )

        # ====================================================
        # LIMITAR HISTORIAL
        # ====================================================

        if len(historial_conversacion) > MAX_MENSAJES * 2:

            historial_conversacion = historial_conversacion[
                -MAX_MENSAJES * 2:
            ]

        # ====================================================
        # MEMORIA LARGA
        # ====================================================

        temas_consultados.append({

            "turno": len(temas_consultados) + 1,

            "pregunta": pregunta

        })

        return respuesta

    except Exception as e:

        return f"Error ejecutando agente: {str(e)}"


# ============================================================
# VER MEMORIA
# ============================================================

def ver_memoria():

    print("\n===== MEMORIA AGENTE =====\n")

    print(
        f"Turnos en memoria corta: "
        f"{len(historial_conversacion)//2}"
    )

    print(
        f"Temas registrados: "
        f"{len(temas_consultados)}"
    )

    if temas_consultados:

        print("\nTemas consultados:\n")

        for tema in temas_consultados:

            print(
                f"Turno {tema['turno']}: "
                f"{tema['pregunta'][:60]}"
            )


# ============================================================
# LIMPIAR MEMORIA
# ============================================================

def limpiar_memoria():

    global historial_conversacion
    global temas_consultados

    historial_conversacion = []

    temas_consultados = []

    print("Memoria reiniciada")


# ============================================================
# LISTO
# ============================================================

print("Sistema de memoria configurado")

In [ ]:
# ============================================================
# CELDA 7 - PRUEBA 1
# ============================================================

print("=" * 60)
print("PRUEBA 1: NOTA MINIMA")
print("=" * 60)

pregunta = "Cual es la nota minima para aprobar una asignatura?"

print(f"\nPregunta:\n{pregunta}")

print("\nGenerando respuesta...\n")

respuesta = preguntar_al_agente(pregunta)

print("\n" + "=" * 60)
print("RESPUESTA FINAL")
print("=" * 60)

print(respuesta)

In [ ]:
# ============================================================
# CELDA 8 - PRUEBA DE MEMORIA CONVERSACIONAL
# ============================================================

print("=" * 60)
print("PRUEBA 2: MEMORIA CONVERSACIONAL")
print("=" * 60)

pregunta = """
Y si repruebo esa asignatura,
cuantas veces puedo intentarlo?
"""

print("\nPregunta:")
print(pregunta)

print("\nGenerando respuesta...\n")

respuesta = preguntar_al_agente(pregunta)

print("\n" + "=" * 60)
print("RESPUESTA FINAL")
print("=" * 60)

print(respuesta)

print("\n")
print("(El agente deberia usar el contexto previo)")

In [ ]:
# ============================================================
# CELDA 9 - PLANIFICACION DEL AGENTE
# ============================================================

print("=" * 60)
print("PRUEBA 3: DECISION DE HERRAMIENTA")
print("=" * 60)

# ============================================================
# REINICIAR MEMORIA
# ============================================================

limpiar_memoria()

# ============================================================
# PREGUNTA GENERAL
# ============================================================

pregunta = """
Que diferencia hay entre un tecnico
y un profesional en educacion superior?
"""

print("\nPregunta:")
print(pregunta)

print("\nGenerando respuesta...\n")

# ============================================================
# EJECUTAR AGENTE
# ============================================================

respuesta = preguntar_al_agente(pregunta)

# ============================================================
# MOSTRAR RESPUESTA
# ============================================================

print("\n" + "=" * 60)
print("RESPUESTA FINAL")
print("=" * 60)

print(respuesta)

print("\n")
print("(El agente deberia usar consulta_general)")

In [ ]:
# ============================================================
# CELDA 10 - RECUPERACION SEMANTICA ESPECIFICA
# ============================================================

print("=" * 60)
print("PRUEBA 4: ASISTENCIA Y RIESGO")
print("=" * 60)

pregunta = """
Cuanto porcentaje de asistencia necesito
para no quedar en riesgo?
"""

print("\nPregunta:")
print(pregunta)

print("\nGenerando respuesta...\n")

# ============================================================
# EJECUTAR AGENTE
# ============================================================

respuesta = preguntar_al_agente(pregunta)

# ============================================================
# MOSTRAR RESPUESTA
# ============================================================

print("\n" + "=" * 60)
print("RESPUESTA FINAL")
print("=" * 60)

print(respuesta)

In [ ]:
# ============================================================
# CELDA 11 - ESTADO DE MEMORIA
# ============================================================

print("=" * 60)
print("ESTADO DE LA MEMORIA DEL AGENTE")
print("=" * 60)

print("\nMostrando memoria actual...\n")

ver_memoria()

In [ ]:
# ============================================================
# CELDA 12 - CHAT INTERACTIVO
# ============================================================

def chat_con_memoria():

    """
    Chat interactivo con memoria conversacional.

    Comandos:
      salir   -> termina conversacion
      memoria -> muestra memoria actual
      limpiar -> reinicia memoria
    """

    print("\n" + "=" * 60)
    print("AGENTE ACADEMICO DUOC UC")
    print("Sistema RAG + Memoria Conversacional")
    print("=" * 60)

    print("\nComandos disponibles:")
    print("  salir   -> terminar chat")
    print("  memoria -> ver memoria")
    print("  limpiar -> reiniciar memoria")

    print("\nEscribe tu pregunta.\n")

    # ========================================================
    # LOOP PRINCIPAL
    # ========================================================

    while True:

        try:

            pregunta = input("\nTu pregunta: ").strip()

            # =================================================
            # VALIDACIONES
            # =================================================

            if not pregunta:

                print("Escribe una pregunta valida.")
                continue

            comando = pregunta.lower()

            # =================================================
            # SALIR
            # =================================================

            if comando == "salir":

                print("\nSesion finalizada.")
                break

            # =================================================
            # VER MEMORIA
            # =================================================

            elif comando == "memoria":

                print()

                ver_memoria()

                continue

            # =================================================
            # LIMPIAR MEMORIA
            # =================================================

            elif comando == "limpiar":

                limpiar_memoria()

                continue

            # =================================================
            # PREGUNTA NORMAL
            # =================================================

            print("\nBuscando respuesta...\n")

            respuesta = preguntar_al_agente(
                pregunta
            )

            print("=" * 60)
            print("RESPUESTA")
            print("=" * 60)

            print(respuesta)

        except KeyboardInterrupt:

            print("\n\nChat interrumpido.")
            break

        except Exception as e:

            print(f"\nError: {str(e)}")


# ============================================================
# INICIAR CHAT
# ============================================================

chat_con_memoria()